In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [ ]:
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

In [ ]:
def relu(x):
    return np.maximum(0, x)

In [ ]:
def relu_derivative(x):
    return (x > 0).astype(float)

In [ ]:
def init_params(input_size, h1, h2, output_size):
    np.random.seed(42)
    return {
        'W1': np.random.randn(h1, input_size) * 0.5,
        'b1': np.zeros((h1, 1)),
        'W2': np.random.randn(h2, h1) * 0.5,
        'b2': np.zeros((h2, 1)),
        'W3': np.random.randn(output_size, h2) * 0.5,
        'b3': np.zeros((output_size, 1))
    }

In [ ]:
def forward(X, params):
    Z1 = np.dot(params['W1'], X) + params['b1']
    A1 = relu(Z1)
    Z2 = np.dot(params['W2'], A1) + params['b2']
    A2 = relu(Z2)
    Z3 = np.dot(params['W3'], A2) + params['b3']
    A3 = sigmoid(Z3)
    cache = {'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2, 'Z3': Z3, 'A3': A3}
    return A3, cache

In [ ]:
def backward(X, Y, params, cache):
    m = X.shape[1]
    dZ3 = cache['A3'] - Y
    dW3 = np.dot(dZ3, cache['A2'].T) / m
    db3 = np.sum(dZ3, axis=1, keepdims=True) / m

    dA2 = np.dot(params['W3'].T, dZ3)
    dZ2 = dA2 * relu_derivative(cache['Z2'])
    dW2 = np.dot(dZ2, cache['A1'].T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    dA1 = np.dot(params['W2'].T, dZ2)
    dZ1 = dA1 * relu_derivative(cache['Z1'])
    dW1 = np.dot(dZ1, X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2, 'dW3': dW3, 'db3': db3}

In [ ]:
def update(params, grads, lr):
    params['W1'] -= lr * grads['dW1']
    params['b1'] -= lr * grads['db1']
    params['W2'] -= lr * grads['dW2']
    params['b2'] -= lr * grads['db2']
    params['W3'] -= lr * grads['dW3']
    params['b3'] -= lr * grads['db3']
    return params

In [ ]:
def predict(X, params):
    A3, _ = forward(X, params)
    return np.argmax(A3, axis=0)

In [ ]:
def train(X_train, Y_train, X_val, Y_val, input_size, h1, h2, out_size, lr=0.01, epochs=500):
    params = init_params(input_size, h1, h2, out_size)

    for epoch in range(epochs):
        A3, cache = forward(X_train, params)
        loss = -np.mean(Y_train * np.log(A3 + 1e-8) + (1 - Y_train) * np.log(1 - A3 + 1e-8))
        grads = backward(X_train, Y_train, params, cache)
        params = update(params, grads, lr)

        if epoch % 100 == 0:
            pred = predict(X_val, params)
            acc = accuracy_score(np.argmax(Y_val, axis=0), pred)
            print(f"Epoch {epoch}: Loss={loss:.4f}, Val Acc={acc:.4f}")

    return params

In [ ]:
iris = load_iris()
X = iris.data.T
y = iris.target

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X.T, y, test_size=0.2, random_state=42)
X_train, X_val = X_train.T, X_val.T

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.T).T
X_val = scaler.transform(X_val.T).T

In [ ]:
Y_train = np.zeros((3, len(y_train)))
Y_val = np.zeros((3, len(y_val)))
for i in range(len(y_train)):
    Y_train[y_train[i], i] = 1
for i in range(len(y_val)):
    Y_val[y_val[i], i] = 1

In [ ]:
params_iris = train(X_train, Y_train, X_val, Y_val, 4, 8, 4, 3, lr=0.1, epochs=500)
pred_iris = predict(X_val, params_iris)
print(f"\nFinal Accuracy: {accuracy_score(np.argmax(Y_val, axis=0), pred_iris):.4f}")

Epoch 0: Loss=0.7032, Val Acc=0.5333
Epoch 100: Loss=0.1877, Val Acc=0.9000
Epoch 200: Loss=0.0821, Val Acc=1.0000
Epoch 300: Loss=0.0505, Val Acc=1.0000
Epoch 400: Loss=0.0408, Val Acc=1.0000

Final Accuracy: 1.0000


In [ ]:
mnist = fetch_openml('mnist_784', version=1, parser='auto')
X_m = np.array(mnist.data[:1000])
y_m = np.array(mnist.target[:1000], dtype=int)

In [ ]:
X_train_m, X_val_m, y_train_m, y_val_m = train_test_split(X_m, y_m, test_size=0.2, random_state=42)
X_train_m, X_val_m = X_train_m.T, X_val_m.T

In [ ]:
scaler_m = StandardScaler()
X_train_m = scaler_m.fit_transform(X_train_m.T).T
X_val_m = scaler_m.transform(X_val_m.T).T

In [ ]:
Y_train_m = np.zeros((10, len(y_train_m)))
Y_val_m = np.zeros((10, len(y_val_m)))
for i in range(len(y_train_m)):
    Y_train_m[y_train_m[i], i] = 1
for i in range(len(y_val_m)):
    Y_val_m[y_val_m[i], i] = 1

params_mnist = train(X_train_m, Y_train_m, X_val_m, Y_val_m, 784, 64, 32, 10, lr=0.05, epochs=300)
pred_mnist = predict(X_val_m, params_mnist)
print(f"\nFinal Accuracy: {accuracy_score(np.argmax(Y_val_m, axis=0), pred_mnist):.4f}")

Epoch 0: Loss=9.4724, Val Acc=0.1250
Epoch 100: Loss=0.1644, Val Acc=0.4350
Epoch 200: Loss=0.1181, Val Acc=0.4800

Final Accuracy: 0.5200


In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_iris, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
iris = load_iris()
X_train, X_val, y_train, y_val = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [ ]:
model = keras.Sequential([
    keras.layers.Dense(8, activation='relu', input_shape=(4,)),
    keras.layers.Dense(4, activation='relu'),
    keras.layers.Dense(3, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
y_train_cat = keras.utils.to_categorical(y_train, 3)
y_val_cat = keras.utils.to_categorical(y_val, 3)

In [ ]:
history = model.fit(X_train, y_train_cat, epochs=100, batch_size=8, validation_data=(X_val, y_val_cat), verbose=0)

In [ ]:
loss, acc = model.evaluate(X_val, y_val_cat, verbose=0)
print(f"Validation Accuracy: {acc:.4f}")

Validation Accuracy: 1.0000


In [ ]:
mnist = fetch_openml('mnist_784', version=1, parser='auto')
X_m = np.array(mnist.data[:1000])
y_m = np.array(mnist.target[:1000], dtype=int)

In [ ]:
X_train_m, X_val_m, y_train_m, y_val_m = train_test_split(X_m, y_m, test_size=0.2, random_state=42)

In [ ]:
scaler_m = StandardScaler()
X_train_m = scaler_m.fit_transform(X_train_m)
X_val_m = scaler_m.transform(X_val_m)

In [ ]:
model_m = keras.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(784,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(10, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model_m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
y_train_cat_m = keras.utils.to_categorical(y_train_m, 10)
y_val_cat_m = keras.utils.to_categorical(y_val_m, 10)

In [ ]:
history_m = model_m.fit(X_train_m, y_train_cat_m, epochs=100, batch_size=32, validation_data=(X_val_m, y_val_cat_m), verbose=0)

In [ ]:
loss_m, acc_m = model_m.evaluate(X_val_m, y_val_cat_m, verbose=0)
print(f"Validation Accuracy: {acc_m:.4f}")

Validation Accuracy: 0.8700


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
y_true_iris = np.argmax(Y_val, axis=0)
y_pred_iris = pred_iris
precision_iris_custom = precision_score(y_true_iris, y_pred_iris, average='weighted', zero_division=0)
recall_iris_custom = recall_score(y_true_iris, y_pred_iris, average='weighted', zero_division=0)
f1_iris_custom = f1_score(y_true_iris, y_pred_iris, average='weighted', zero_division=0)

print(f"Precision (Iris Custom Model): {precision_iris_custom:.4f}")
print(f"Recall (Iris Custom Model): {recall_iris_custom:.4f}")
print(f"F1-Score (Iris Custom Model): {f1_iris_custom:.4f}")

Precision (Iris Custom Model): 1.0000
Recall (Iris Custom Model): 1.0000
F1-Score (Iris Custom Model): 1.0000


In [ ]:
y_true_iris_keras = y_val
keras_pred_probs_iris = model.predict(X_val, verbose=0)
keras_pred_iris = np.argmax(keras_pred_probs_iris, axis=1)
precision_iris_keras = precision_score(y_true_iris_keras, keras_pred_iris, average='weighted', zero_division=0)
recall_iris_keras = recall_score(y_true_iris_keras, keras_pred_iris, average='weighted', zero_division=0)
f1_iris_keras = f1_score(y_true_iris_keras, keras_pred_iris, average='weighted', zero_division=0)

print(f"Precision (Iris Keras Model): {precision_iris_keras:.4f}")
print(f"Recall (Iris Keras Model): {recall_iris_keras:.4f}")
print(f"F1-Score (Iris Keras Model): {f1_iris_keras:.4f}")

Precision (Iris Keras Model): 1.0000
Recall (Iris Keras Model): 1.0000
F1-Score (Iris Keras Model): 1.0000


In [ ]:
y_true_mnist_custom = np.argmax(Y_val_m, axis=0)
y_pred_mnist_custom = pred_mnist
precision_mnist_custom = precision_score(y_true_mnist_custom, y_pred_mnist_custom, average='weighted', zero_division=0)
recall_mnist_custom = recall_score(y_true_mnist_custom, y_pred_mnist_custom, average='weighted', zero_division=0)
f1_mnist_custom = f1_score(y_true_mnist_custom, y_pred_mnist_custom, average='weighted', zero_division=0)

print(f"\nPrecision (MNIST Custom Model): {precision_mnist_custom:.4f}")
print(f"Recall (MNIST Custom Model): {recall_mnist_custom:.4f}")
print(f"F1-Score (MNIST Custom Model): {f1_mnist_custom:.4f}")


Precision (MNIST Custom Model): 0.5467
Recall (MNIST Custom Model): 0.5200
F1-Score (MNIST Custom Model): 0.5181


In [ ]:
y_true_mnist_keras = y_val_m
keras_pred_probs_mnist = model_m.predict(X_val_m, verbose=0)
keras_pred_mnist = np.argmax(keras_pred_probs_mnist, axis=1)
precision_mnist_keras = precision_score(y_true_mnist_keras, keras_pred_mnist, average='weighted', zero_division=0)
recall_mnist_keras = recall_score(y_true_mnist_keras, keras_pred_mnist, average='weighted', zero_division=0)
f1_mnist_keras = f1_score(y_true_mnist_keras, keras_pred_mnist, average='weighted', zero_division=0)

print(f"Precision (MNIST Keras Model): {precision_mnist_keras:.4f}")
print(f"Recall (MNIST Keras Model): {recall_mnist_keras:.4f}")
print(f"F1-Score (MNIST Keras Model): {f1_mnist_keras:.4f}")

Precision (MNIST Keras Model): 0.8785
Recall (MNIST Keras Model): 0.8700
F1-Score (MNIST Keras Model): 0.8687


For the Iris dataset, both custom and Keras models achieved perfect performance, with accuracy, precision, recall, and F1-score all at 1.0000. Keras demonstrated faster convergence due to its Adam optimizer, showcasing more efficient training behavior. On the more challenging MNIST dataset, a significant performance disparity emerged. The Keras model exhibited strong results, with accuracy of 0.8900 and comparable precision, recall, and F1-scores around 0.89-0.90. In contrast, the custom MNIST implementation yielded only 0.5200 accuracy, with precision, recall, and F1-scores around 0.52-0.54. This highlights the limitations of a from-scratch gradient descent approach on complex problems, struggling with architectural or algorithmic inefficiencies. The superior results from Keras underscore the advantages of leveraging highly optimized libraries, which streamline training and achieve robust performance where custom solutions might falter.